In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(r"D:\works\PE\code\FCshaixuan.csv", encoding="ISO-8859-1")

category = df.iloc[:, 0]

y = df.iloc[:, 1]

X = df.iloc[:, 2:]

data = pd.concat([category, y, X], axis=1)

categories = ["Co", "Fe", "Cr", "Ni", "Ti", "Zr", "Hf"]

train_list = []
test_list = []

for cat in categories:
    group = data[data.iloc[:, 0] == cat]  
    
    train_part, test_part = train_test_split(group, test_size=0.2, random_state=42)
    
    train_list.append(train_part)
    test_list.append(test_part)


train_data = pd.concat(train_list, axis=0).sample(frac=1, random_state=42)  
test_data = pd.concat(test_list, axis=0).sample(frac=1, random_state=42)

category_train = train_data.iloc[:, 0]
y_train = train_data.iloc[:, 1]
X_train = train_data.iloc[:, 2:]

category_test = test_data.iloc[:, 0]
y_test = test_data.iloc[:, 1]
X_test = test_data.iloc[:, 2:]

print("Train set size:", X_train.shape, "Test set size:", X_test.shape)
print("Train category distribution:\n", category_train.value_counts())
print("Test category distribution:\n", category_test.value_counts())

print("===== Top n samples of Training Set =====")
for i in range(2):
    print("X_train:", X_train.iloc[i].values, " | y_train:", y_train.iloc[i])

print("\n===== Top n samples of Test Set =====")
for i in range(1):
    print("X_test:", X_test.iloc[i].values, " | y_test:", y_test.iloc[i])


训练集大小: (402, 118) 测试集大小: (105, 118)
训练类别分布:
 center
Ti    122
Fe    102
Co     88
Zr     53
Cr     16
Hf     11
Ni     10
Name: count, dtype: int64
测试类别分布:
 center
Ti    31
Fe    26
Co    23
Zr    14
Cr     5
Hf     3
Ni     3
Name: count, dtype: int64
===== 训练集前n条 =====
X_train: [ 1.25000000e+03  2.50000000e+01  9.99999730e-01  1.00000000e+01
  1.90212766e+01  1.01069264e+01  4.15424232e+01  2.40214610e+01
  1.03203000e+01  1.30000000e+01  7.84838321e+09  8.00000000e+00
  1.75616602e+01  3.35898565e+01  2.76393537e+01  2.00000000e+00
  0.00000000e+00  3.45453475e+01  1.14990237e+01  2.52466777e+01
  1.56732407e+01  0.00000000e+00  8.28591905e+00  7.22319589e+00
  1.85443349e+01  2.44800000e+01  7.02995586e+01  1.24292012e+01
  2.61628428e+01  2.92362535e+01  0.00000000e+00  1.11492757e+01
  3.63982024e+01  2.76391985e+02  2.74766202e+02 -8.19255480e+00
  0.00000000e+00  1.14990237e+01 -1.92050035e+00  2.31152022e+00
  4.24138852e+01  6.12494324e+00  6.63676624e+00  2.00000000e+00
  2.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ===========================
# Data Preprocessing
# ===========================
# Convert to numpy format
X_train_np = np.array(X_train)
X_test_np = np.array(X_test)
y_train_np = np.array(y_train).ravel()
y_test_np = np.array(y_test).ravel()

# Standardization (CNN is sensitive to feature scales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_np)
X_test_scaled = scaler.transform(X_test_np)

# CNN input requires 3D: (n_samples, n_features, n_channels)
X_train_cnn = np.expand_dims(X_train_scaled, axis=-1)
X_test_cnn = np.expand_dims(X_test_scaled, axis=-1)

print("X_train_cnn shape:", X_train_cnn.shape)
print("X_test_cnn shape:", X_test_cnn.shape)

X_train_cnn shape: (402, 118, 1)
X_test_cnn shape: (105, 118, 1)


In [ ]:
# Load the model
best_model = tf.keras.models.load_model("best_cnn_regression.keras")

y_pred = best_model.predict(X_test_cnn).ravel()
y_train_cnn_pred = best_model.predict(X_train_cnn).ravel()

# Test set metrics
r2_test = r2_score(y_test_np, y_pred)
mse_test = mean_squared_error(y_test_np, y_pred)
mae_test = mean_absolute_error(y_test_np, y_pred)

# Train set metrics
r2_train = r2_score(y_train_np, y_train_cnn_pred)
mse_train = mean_squared_error(y_train_np, y_train_cnn_pred)
mae_train = mean_absolute_error(y_train_np, y_train_cnn_pred)

# Output results
print("\nModel Evaluation Results:")
print("Test Set:")
print(f"  R²:  {r2_test:.4f}")
print(f"  MSE: {mse_test:.4f}")
print(f"  MAE: {mae_test:.4f}")

print("\nTrain Set:")
print(f"  R²:  {r2_train:.4f}")
print(f"  MSE: {mse_train:.4f}")
print(f"  MAE: {mae_train:.4f}")

13/13 [==============================] - 0s 3ms/step

模型评估结果：
测试集：
  R²:  0.3336
  MSE: 0.0168
  MAE: 0.1087

训练集：
  R²:  0.3764
  MSE: 0.0160
  MAE: 0.1063
